# Предсказание зарплаты разными вариантами градиентного бустинга

В этой части будем предсказывать зарплату data scientist-ов в зависимости  от ряда факторов с помощью градиентного бустинга.

В датасете есть следующие признаки:



* work_year: The number of years of work experience in the field of data science.

* experience_level: The level of experience, such as Junior, Senior, or Lead.

* employment_type: The type of employment, such as Full-time or Contract.

* job_title: The specific job title or role, such as Data Analyst or Data Scientist.

* salary: The salary amount for the given job.

* salary_currency: The currency in which the salary is denoted.

* salary_in_usd: The equivalent salary amount converted to US dollars (USD) for comparison purposes.

* employee_residence: The country or region where the employee resides.

* remote_ratio: The percentage of remote work offered in the job.

* company_location: The location of the company or organization.

* company_size: The company's size is categorized as Small, Medium, or Large.

In [71]:
!pip install numpy==1.26.4

In [1]:
import pandas as pd
url = "https://github.com/hse-ds/iad-intro-ds/blob/master/2024/homeworks/hw08_boosting_clustering/ds_salaries.csv?raw=true"

df = pd.read_csv(url)

df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


## Задание 1 (0.5 балла) Подготовка



*   Разделите выборку на train, val, test (80%, 10%, 10%)
*   Выдерите salary_in_usd в качестве таргета
*   Найдите и удалите признак, из-за которого возможен лик в данных


Признаки salary и salary_in_usd подозрительно похожи, поэтому salary мы удалим

In [2]:
df.drop('salary', inplace=True, axis=1)

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop('salary_in_usd', axis=1)
y = df['salary_in_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X, y, test_size=0.5, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape)

(3004, 9) (1878, 9) (1877, 9)


## Задание 2 (0.5 балла) Линейная модель


*   Закодируйте категориальные  признаки с помощью OneHotEncoder
*   Обучите модель линейной регрессии
*   Оцените  качество через MAPE и RMSE


In [4]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3755 entries, 0 to 3754
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   work_year           3755 non-null   int64 
 1   experience_level    3755 non-null   object
 2   employment_type     3755 non-null   object
 3   job_title           3755 non-null   object
 4   salary_currency     3755 non-null   object
 5   employee_residence  3755 non-null   object
 6   remote_ratio        3755 non-null   int64 
 7   company_location    3755 non-null   object
 8   company_size        3755 non-null   object
dtypes: int64(2), object(7)
memory usage: 264.2+ KB


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder

categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoder.fit(X_train[categorical_features])

X_train_encoded = encoder.transform(X_train[categorical_features])
X_val_encoded = encoder.transform(X_val[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)
X_val_encoded_df = pd.DataFrame(
    X_val_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_val.index
)
X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)

X_train_numeric = X_train.drop(columns=categorical_features)
X_val_numeric = X_val.drop(columns=categorical_features)
X_test_numeric = X_test.drop(columns=categorical_features)

X_train = pd.concat([X_train_numeric, X_train_encoded_df], axis=1)
X_val = pd.concat([X_val_numeric, X_val_encoded_df], axis=1)
X_test = pd.concat([X_test_numeric, X_test_encoded_df], axis=1)

In [6]:
print(X_train.shape, X_val.shape, X_test.shape)

(3004, 259) (1878, 259) (1877, 259)


In [7]:
import numpy as np

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'MAPE: {mape:.4f}')
print(f'RMSE: {rmse:.4f}')

MAPE: 0.3086
RMSE: 44007.5426


## Задание 3 (0.5 балла) XGboost

Начнем с библиотеки xgboost.

Обучите модель `XGBRegressor` на тех же данных, что линейную модель, подобрав оптимальные гиперпараметры (`max_depth, learning_rate, n_estimators, gamma`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [8]:
!pip install optuna -q

In [9]:
from xgboost import XGBRegressor
import optuna

def objective(trial):
  params = {
        "max_depth": trial.suggest_int("max_depth", 1, 30),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1200),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "tree_method": "auto",
        "early_stopping_rounds" : 20

  }

  model = XGBRegressor(**params)
  model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

  y_val_pred = model.predict(X_val)

  rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

  return rmse

study = optuna.create_study(direction='minimize')

study.optimize(objective, n_trials=50)

print("Наилучшие гиперпараметры:", study.best_params)
print("Наилучшее значение RMSE:", study.best_value)

[I 2025-04-04 14:44:17,374] A new study created in memory with name: no-name-6c85242b-c16f-4708-9267-390aba879bc9
[I 2025-04-04 14:44:23,457] Trial 0 finished with value: 45568.76263406765 and parameters: {'max_depth': 14, 'learning_rate': 0.16495024802169994, 'n_estimators': 1048, 'gamma': 3.5424095875152872, 'min_child_weight': 8, 'subsample': 0.5422618537344837, 'colsample_bytree': 0.7663136858266386, 'reg_alpha': 3.9496200290964096, 'reg_lambda': 1.6423190851294933}. Best is trial 0 with value: 45568.76263406765.
[I 2025-04-04 14:44:31,473] Trial 1 finished with value: 46921.90345670133 and parameters: {'max_depth': 30, 'learning_rate': 0.22116638709283284, 'n_estimators': 1123, 'gamma': 3.7062852607159846, 'min_child_weight': 6, 'subsample': 0.25466421264300143, 'colsample_bytree': 0.5210595869228334, 'reg_alpha': 0.33148009299379444, 'reg_lambda': 2.594584257838575}. Best is trial 0 with value: 45568.76263406765.
[I 2025-04-04 14:44:44,496] Trial 2 finished with value: 47834.3595

Наилучшие гиперпараметры: {'max_depth': 23, 'learning_rate': 0.045659939189782894, 'n_estimators': 506, 'gamma': 1.0724901061145402, 'min_child_weight': 1, 'subsample': 0.9989093878756179, 'colsample_bytree': 0.6408158272985836, 'reg_alpha': 1.1029225820711748, 'reg_lambda': 1.2561419679587205}
Наилучшее значение RMSE: 43957.489282259965


In [10]:
import time

best_params = study.best_params

xgbmodel = XGBRegressor(**best_params, random_state=42)

start_train_time = time.time()
xgbmodel.fit(X_train, y_train)
end_train_time = time.time() - start_train_time

start_predict_time = time.time()
y_pred = xgbmodel.predict(X_test)
end_predict_time = time.time() - start_predict_time

print(f"Train time = {end_train_time:.4f}")
print(f"Predict time = {end_predict_time:.4f}")

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'MAPE: {mape:.4f}')
print(f'RMSE: {rmse:.4f}')

Train time = 20.0276
Predict time = 0.5019
MAPE: 0.2133
RMSE: 38803.0597


Любопытно, что наилучшие параметры достигаются при очень глубоких деревьях (23), а прирост RMSE очень небольшой

## Задание 4 (1 балл) CatBoost

Теперь библиотека CatBoost.

Обучите модель `CatBoostRegressor`, подобрав оптимальные гиперпараметры (`depth, learning_rate, iterations`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [11]:
!pip install catboost -q

In [13]:
from catboost import CatBoostRegressor

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 1, 16),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-5, 10, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-5, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 30),
        'verbose': False,
        'random_state': 42
    }

    model = CatBoostRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        verbose=False
    )

    y_val_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("Наилучшие гиперпараметры:", study.best_params)
print("Наилучшее значение RMSE:", study.best_value)

[I 2025-04-04 14:49:50,700] A new study created in memory with name: no-name-fe4331e7-cd9a-4674-9c89-fd66186a5dec
[I 2025-04-04 14:49:52,291] Trial 0 finished with value: 45390.77503002662 and parameters: {'iterations': 121, 'learning_rate': 0.03661951039526591, 'depth': 8, 'l2_leaf_reg': 0.00949075065117338, 'random_strength': 0.04108320913566069, 'bagging_temperature': 0.8089215049701525, 'grow_policy': 'Depthwise', 'min_data_in_leaf': 10}. Best is trial 0 with value: 45390.77503002662.
[I 2025-04-04 14:49:56,952] Trial 1 finished with value: 45667.689444358 and parameters: {'iterations': 425, 'learning_rate': 0.010554414360090945, 'depth': 12, 'l2_leaf_reg': 2.0580446656081763, 'random_strength': 0.15404886475063528, 'bagging_temperature': 0.573511885848339, 'grow_policy': 'Lossguide', 'min_data_in_leaf': 2}. Best is trial 0 with value: 45390.77503002662.
[I 2025-04-04 14:50:01,430] Trial 2 finished with value: 44595.26229866048 and parameters: {'iterations': 971, 'learning_rate': 0

Наилучшие гиперпараметры: {'iterations': 299, 'learning_rate': 0.14028307489017253, 'depth': 14, 'l2_leaf_reg': 0.00042958009099308025, 'random_strength': 0.003949299336620266, 'bagging_temperature': 0.9981862095963142, 'grow_policy': 'SymmetricTree', 'min_data_in_leaf': 30}
Наилучшее значение RMSE: 43757.796322761336


In [14]:
best_params = study.best_params

cbmodel = CatBoostRegressor(**best_params, random_state=42)

start_train_time = time.time()
cbmodel.fit(X_train, y_train, verbose=False)
end_train_time = time.time() - start_train_time

start_predict_time = time.time()
y_pred = cbmodel.predict(X_test)
end_predict_time = time.time() - start_predict_time

print(f"Train time = {end_train_time:.4f}")
print(f"Predict time = {end_predict_time:.4f}")

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'MAPE: {mape:.4f}')
print(f'RMSE: {rmse:.4f}')

0:	learn: 58875.8558021	total: 91.9ms	remaining: 27.4s
1:	learn: 55521.2953824	total: 199ms	remaining: 29.5s
2:	learn: 52895.7698776	total: 289ms	remaining: 28.5s
3:	learn: 50819.7585769	total: 380ms	remaining: 28.1s
4:	learn: 49193.2137168	total: 475ms	remaining: 27.9s
5:	learn: 47708.5950544	total: 566ms	remaining: 27.6s
6:	learn: 46663.6430957	total: 666ms	remaining: 27.8s
7:	learn: 45746.3750060	total: 760ms	remaining: 27.6s
8:	learn: 45024.2059628	total: 850ms	remaining: 27.4s
9:	learn: 44525.2659717	total: 945ms	remaining: 27.3s
10:	learn: 44064.7432763	total: 1.04s	remaining: 27.1s
11:	learn: 43690.6317055	total: 1.13s	remaining: 27s
12:	learn: 43310.9138443	total: 1.24s	remaining: 27.2s
13:	learn: 43054.5623215	total: 1.33s	remaining: 27.1s
14:	learn: 42841.7417050	total: 1.42s	remaining: 26.9s
15:	learn: 42660.5778949	total: 1.52s	remaining: 26.8s
16:	learn: 42434.7889111	total: 1.63s	remaining: 27s
17:	learn: 42283.8281496	total: 1.72s	remaining: 26.8s
18:	learn: 42148.352149

Снова глубокие деревья и снова небольшой прирост метрик

Для применения catboost моделей не обязательно сначала кодировать категориальные признаки, модель может кодировать их сама. Обучите catboost с подбором оптимальных гиперпараметров снова, используя pool для передачи данных в модель с указанием какие признаки категориальные, а какие нет с помощью параметра cat_features. Оцените качество и время. Стало ли лучше?

In [15]:
X = df.drop('salary_in_usd', axis=1)
y = df['salary_in_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X, y, test_size=0.5, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape)

(3004, 9) (1878, 9) (1877, 9)


In [16]:
from catboost import Pool

categorical_features_indices = [i for i, col in enumerate(X_train.columns) if X_train[col].dtype == 'object']

train_pool = Pool(X_train, label=y_train, cat_features=categorical_features_indices)
val_pool = Pool(X_val, label=y_val, cat_features=categorical_features_indices)
test_pool = Pool(X_test, label=y_test, cat_features=categorical_features_indices)

In [17]:
from catboost import CatBoostRegressor

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 1, 16),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-5, 10, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-5, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 30),
        'verbose': False,
        'random_state': 42
    }

    model = CatBoostRegressor(**params)
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
        verbose=False
    )

    y_val_pred = model.predict(val_pool)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("Наилучшие гиперпараметры:", study.best_params)
print("Наилучшее значение RMSE:", study.best_value)

[I 2025-04-04 14:57:25,626] A new study created in memory with name: no-name-ab0839bc-f465-420a-82f8-6b0e953047ae
[I 2025-04-04 14:57:26,289] Trial 0 finished with value: 48941.33453728454 and parameters: {'iterations': 556, 'learning_rate': 0.04240303422528602, 'depth': 1, 'l2_leaf_reg': 0.11378748653601627, 'random_strength': 2.2658270711019357, 'bagging_temperature': 0.9757070410064898, 'grow_policy': 'Lossguide', 'min_data_in_leaf': 14}. Best is trial 0 with value: 48941.33453728454.
[I 2025-04-04 14:57:30,673] Trial 1 finished with value: 46931.36857785034 and parameters: {'iterations': 404, 'learning_rate': 0.011653685660239409, 'depth': 7, 'l2_leaf_reg': 1.7609154113390408e-05, 'random_strength': 1.672447056428799e-05, 'bagging_temperature': 0.16690530831758799, 'grow_policy': 'SymmetricTree', 'min_data_in_leaf': 7}. Best is trial 1 with value: 46931.36857785034.
[I 2025-04-04 14:57:31,347] Trial 2 finished with value: 48385.2352870282 and parameters: {'iterations': 276, 'learni

Наилучшие гиперпараметры: {'iterations': 881, 'learning_rate': 0.06693662635379491, 'depth': 11, 'l2_leaf_reg': 0.06509294118862317, 'random_strength': 0.00023382528541152138, 'bagging_temperature': 0.7445901680766155, 'grow_policy': 'SymmetricTree', 'min_data_in_leaf': 25}
Наилучшее значение RMSE: 45904.55141933032


In [19]:
best_params = study.best_params

cbmodel = CatBoostRegressor(**best_params, random_state=42)

start_train_time = time.time()
cbmodel.fit(train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
        verbose=False
    )
end_train_time = time.time() - start_train_time

start_predict_time = time.time()
y_pred = cbmodel.predict(X_test)
end_predict_time = time.time() - start_predict_time

print(f"Train time = {end_train_time:.4f}")
print(f"Predict time = {end_predict_time:.4f}")

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'MAPE: {mape:.4f}')
print(f'RMSE: {rmse:.4f}')

Train time = 10.6695
Predict time = 0.0073
MAPE: 0.3213
RMSE: 42627.8995


MAPE стало получше, а вот RMSE заметно поплохело

## Задание 5 (0.5 балла) LightGBM

И наконец библиотека LightGBM - используйте `LGBMRegressor`, снова подберите гиперпараметры, оцените качество и скорость.


In [20]:
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoder.fit(X_train[categorical_features])

X_train_encoded = encoder.transform(X_train[categorical_features])
X_val_encoded = encoder.transform(X_val[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)
X_val_encoded_df = pd.DataFrame(
    X_val_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_val.index
)
X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)

X_train_numeric = X_train.drop(columns=categorical_features)
X_val_numeric = X_val.drop(columns=categorical_features)
X_test_numeric = X_test.drop(columns=categorical_features)

X_train = pd.concat([X_train_numeric, X_train_encoded_df], axis=1)
X_val = pd.concat([X_val_numeric, X_val_encoded_df], axis=1)
X_test = pd.concat([X_test_numeric, X_test_encoded_df], axis=1)

In [23]:
from lightgbm import LGBMRegressor

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        'random_state': 42,
        'verbosity': -1
    }

    model = LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
    )

    y_val_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("Наилучшие гиперпараметры:", study.best_params)
print("Наилучшее значение RMSE:", study.best_value)

[I 2025-04-04 15:19:28,906] A new study created in memory with name: no-name-b34a13e0-def2-4633-8cee-9d2c9874b689
[I 2025-04-04 15:19:29,212] Trial 0 finished with value: 48165.30230006859 and parameters: {'n_estimators': 314, 'learning_rate': 0.16727804089595247, 'max_depth': 7, 'num_leaves': 60, 'min_child_samples': 79, 'subsample': 0.9531942423913948, 'colsample_bytree': 0.8537696429144233, 'reg_alpha': 1.7783639558889541, 'reg_lambda': 2.897622250130122}. Best is trial 0 with value: 48165.30230006859.
[I 2025-04-04 15:19:29,570] Trial 1 finished with value: 47958.79984753422 and parameters: {'n_estimators': 357, 'learning_rate': 0.014611313478822474, 'max_depth': 14, 'num_leaves': 68, 'min_child_samples': 37, 'subsample': 0.656698776557385, 'colsample_bytree': 0.6992599984763614, 'reg_alpha': 3.3706529871676896, 'reg_lambda': 5.78788385929665}. Best is trial 1 with value: 47958.79984753422.
[I 2025-04-04 15:19:29,874] Trial 2 finished with value: 48229.84314745871 and parameters: {

Наилучшие гиперпараметры: {'n_estimators': 342, 'learning_rate': 0.144856658845347, 'max_depth': 15, 'num_leaves': 41, 'min_child_samples': 5, 'subsample': 0.9523303663785093, 'colsample_bytree': 0.7035970819871655, 'reg_alpha': 4.2767338369691075, 'reg_lambda': 6.173876263769651}
Наилучшее значение RMSE: 44946.05176852948


In [24]:
best_params = study.best_params

lgbmodel = LGBMRegressor(**best_params, random_state=42)

start_train_time = time.time()
lgbmodel.fit(X_train, y_train)
end_train_time = time.time() - start_train_time

start_predict_time = time.time()
y_pred = lgbmodel.predict(X_test)
end_predict_time = time.time() - start_predict_time

print(f"Train time = {end_train_time:.4f}")
print(f"Predict time = {end_predict_time:.4f}")

mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'MAPE: {mape:.4f}')
print(f'RMSE: {rmse:.4f}')

Train time = 0.2876
Predict time = 0.0764
MAPE: 0.2598
RMSE: 40456.8832


Гораздо быстрее остальных моделей, но качество немного пониже

## Задание 6 (2 балла) Сравнение и выводы

Сравните модели бустинга и сделайте про них выводы, какая из моделей показала лучший/худший результат по качеству, скорости обучения и скорости предсказания? Как отличаются гиперпараметры для разных моделей?

| Модель                  | MAPE     | RMSE       | Время обучения (сек) | Время предсказания (сек) |
|-------------------------|----------|------------|------------------------|---------------------------|
| Линейная регрессия      | 0.3086   | 44007.5426 | —                      | —                         |
| XGBoost                 | 0.2133   | 38803.0597 | 20.0276                | 0.5019                    |
| CatBoost                | 0.2155   | 38807.2691 | 31.6776                | 0.0124                    |
| CatBoost (категориальные)| 0.3213  | 42627.8995 | 10.6695                | 0.0073                    |
| LightGBM                | 0.2598   | 40456.8832 | 0.2876                 | 0.0764                    |


По совокупным характеристикам CatBoost лучше всех, а прямо в спину ему дышит XGBoost. Если дать CatBoost самому расправиться с категориальными переменными, скорость обучения значительно увеличивается, MAPE значительно возрастает, но RMSE заметно плохеет. LightGBM хуже всех из бустингов, но зато на два порядка быстрее. По неведомым причинам все бустинги стремятся к очень глубоким деревьям. Видимо у нас очень сложные данные, на которые слабых деревьев не хватает.